In [4]:
# Lab 6 - Task 2 (Intermediate):

import re
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# =====================================================================
# Chunking functions
# =====================================================================

def word_spans(text):
    return [(m.start(), m.end()) for m in re.finditer(r"\S+", text)]

def windows(text, size=40, overlap=10, offset=0):
    if size <= 0 or not 0 <= overlap < size:
        raise ValueError("Require size > 0 and 0 <= overlap < size")
    spans = word_spans(text)
    result = []
    for i in range(0, len(spans), size - overlap):
        j = min(i + size, len(spans))
        result.append((offset + spans[i][0], offset + spans[j - 1][1]))
        if j == len(spans):
            break
    return result

def sentence_chunks(text, size=40):
    if size <= 0:
        raise ValueError("size must be positive")
    # Teaching splitter: punctuation followed by whitespace.
    cuts = [0] + [m.end() for m in
                  re.finditer(r"[.!?]\s+", text)] + [len(text)]
    out, pending, count = [], None, 0
    for a, b in zip(cuts, cuts[1:]):
        segment = text[a:b]
        n = len(segment.split())
        if not n:
            continue
        if n > size:
            if pending is not None:
                out.append(pending)
            out.extend(windows(segment, size, 0, a))
            pending, count = None, 0
        elif pending is not None and count + n <= size:
            pending, count = (pending[0], b), count + n
        else:
            if pending is not None:
                out.append(pending)
            pending, count = (a, b), n
    if pending is not None:
        out.append(pending)
    return out

def make_chunks(docs, strategy):
    result = []
    for doc_id, text in docs.items():
        spans = (sentence_chunks(text) if strategy == "sentence"
                 else windows(text, 40, int(strategy)))
        for i, (a, b) in enumerate(spans):
            result.append(dict(id=f"{doc_id}:{i}", doc=doc_id,
                               start=a, end=b, text=text[a:b]))
    return result

# =====================================================================
# Corpus: three synthetic documents (150-250 words each).
# =====================================================================
HANDBOOK = (
    "This handbook section explains how coursework is assessed at the "
    "university. Every assignment must be submitted through the learning "
    "portal before the stated deadline, and the portal records the exact "
    "time of submission. Late submission is normally penalized. Illness "
    "extensions require medical evidence before the deadline. A student "
    "who is ill must inform the course coordinator as early as possible "
    "and attach a valid medical certificate with the extension request. "
    "Extensions are not automatic, and the coordinator may refuse a "
    "request that arrives after the deadline has passed. Group "
    "assignments follow the same rule for the whole group, so one "
    "member's illness must be reported by the group leader. All "
    "submitted work must be the student's own work. Copying from another "
    "student, from a website, or from a generated answer without "
    "acknowledgement is treated as plagiarism. Suspected plagiarism is "
    "referred to the disciplinary committee, and penalties range from a "
    "zero mark for the assignment to suspension from the course. "
    "Students may appeal an assessment decision within seven working "
    "days by writing to the examinations office with the assignment "
    "reference number. The examinations office replies to every appeal "
    "in writing, and its decision is final. Students should keep a "
    "personal copy of every submitted file until results are announced.")

IT_SUPPORT = (
    "The IT help desk supports student accounts, email and the learning "
    "portal. Reset a forgotten password using the recovery link. The "
    "recovery link is sent to the student's registered mobile number "
    "and stays valid for thirty minutes. Never share recovery codes. A "
    "recovery code is private, and help desk staff will never ask for "
    "it on a call or in a message. If a student reports that the "
    "recovery message has not arrived after several attempts, the help "
    "desk first checks whether the registered mobile number on the "
    "account is still active, then confirms that the inbox and message "
    "filters are not blocking university messages, and finally raises a "
    "ticket so that the identity team can verify the account documents "
    "before issuing a manual reset during office hours. Students who "
    "change their mobile number must update it at the student affairs "
    "office, because account recovery depends on that number. For "
    "portal problems, students should note the exact error message and "
    "the time it appeared, since these details help the support team "
    "trace the fault. The help desk is open from eight in the morning "
    "until eight in the evening on working days.")

LIBRARY = (
    "The university library lends books, past papers and study guides "
    "to registered students. Each undergraduate may borrow up to three "
    "books at one time, and the standard loan period is fourteen days. "
    "Borrowed books must be returned or renewed before the due date "
    "shown in the library account. A fine is charged for every day a "
    "book stays overdue, and borrowing rights are suspended while a "
    "fine remains unpaid. Reference books, current journals and the "
    "reserved collection cannot be taken out of the library; they may "
    "be read in the reading room during opening hours. Students are "
    "responsible for books issued on their card and must report a lost "
    "card immediately, because charges for items borrowed on a lost "
    "card remain the owner's responsibility until the report is made. "
    "Quiet study rooms can be booked online for a maximum of two hours "
    "per group per day. Food and drinks other than bottled water are "
    "not allowed inside the library, and mobile phones must remain on "
    "silent. The library opens at eight in the morning and closes at "
    "ten at night during the semester.")

RULE = "Late submission is normally penalized."
EXCEPTION = "Illness extensions require medical evidence before the deadline."

if __name__ == "__main__":
    # --- Reference behaviour checks -----------------------------------
    assert windows("") == []
    assert windows("short text") == [(0, 10)]
    try:
        windows("a b", 2, 2)
    except ValueError:
        pass
    else:
        raise AssertionError("Invalid overlap was accepted")
    long = " ".join(["word"] * 95) + "."
    assert all(len(long[a:b].split()) <= 40
               for a, b in sentence_chunks(long))
    example = {"example": "Reset your password. Keep recovery codes private."}
    for strategy in ["0", "10", "sentence"]:
        for c in make_chunks(example, strategy):
            assert c["text"] == example[c["doc"]][c["start"]:c["end"]]
            print(strategy, c)
    print("Reference behaviour checks passed")

    # --- Boundary tests -------------------------------------------------
    print()
    print("Boundary tests:")
    print("empty text -> windows:", windows(""),
          "| sentence:", sentence_chunks(""))
    short = "Only five words are here."
    print("short document -> windows:", windows(short),
          "| sentence:", sentence_chunks(short))
    try:
        windows("a b c", 40, 40)
        print("invalid overlap -> NOT rejected (unexpected)")
    except ValueError as error:
        print("invalid overlap -> ValueError:", error)
    long_sentence = " ".join(["word"] * 95) + "."
    fallback = sentence_chunks(long_sentence)
    print("95-word sentence -> sentence-aware fallback chunks:",
          len(fallback),
          "sizes:", [len(long_sentence[a:b].split()) for a, b in fallback])

    # --- Corpus comparison ----------------------------------------------
    corpus = {"handbook": HANDBOOK, "it_support": IT_SUPPORT,
              "library": LIBRARY}
    print()
    print("Corpus word counts:",
          {doc_id: len(text.split()) for doc_id, text in corpus.items()})
    original_words = sum(len(t.split()) for t in corpus.values())
    print(f"{'strategy':<10} {'chunks':>6} {'min':>4} {'max':>4} "
          f"{'emitted':>8} {'duplicated':>10}  rule+exception together")
    for strategy in ["0", "10", "sentence"]:
        chunks = make_chunks(corpus, strategy)
        sizes = [len(c["text"].split()) for c in chunks]
        emitted = sum(sizes)
        together = any(RULE in c["text"] and EXCEPTION in c["text"]
                       for c in chunks)
        intact = any(EXCEPTION in c["text"] for c in chunks)
        print(f"{strategy:<10} {len(chunks):>6} {min(sizes):>4} "
              f"{max(sizes):>4} {emitted:>8} {emitted - original_words:>10}  "
              f"{together} (exception intact alone: {intact})")

    # --- Where the simple sentence splitter fails -----------------------
    print()
    demo = "Dr. Smith explained the rule, e.g. late work, in class. It applies to everyone."
    print("Abbreviation demo on:", demo)
    cuts = [0] + [m.end() for m in
                  re.finditer(r"[.!?]\s+", demo)] + [len(demo)]
    for a, b in zip(cuts, cuts[1:]):
        print("  raw segment:", repr(demo[a:b]))
    print("The teaching splitter cuts after 'Dr.' and 'e.g.' because it only")
    print("looks for punctuation followed by whitespace; it cannot tell an")
    print("abbreviation from a real sentence ending, so 'Dr.' becomes a")
    print("false segment of its own before the packing step merges text.")

    # --- Semantic suggestions: TF-IDF versus embeddings -----------------
    print()
    print("=" * 70)
    print("SEMANTIC CHECK - paraphrased query on two records")
    print("=" * 70)

    texts = {"it": "A forgotten password can be reset using the recovery link.",
             "assessment": "Assignments are uploaded through the learning portal."}
    records = make_chunks(texts, "sentence")
    query = "How can I regain access to my account?"

    vectorizer = TfidfVectorizer()
    matrix = vectorizer.fit_transform([c["text"] for c in records])
    q = vectorizer.transform([query])
    scores = (q @ matrix.T).toarray()[0] if q.nnz else np.zeros(len(records))
    print("Backend: tfidf")
    for c, s in sorted(zip(records, scores), key=lambda x: -x[1]):
        print(c["id"], f"score={s:.3f}")
        print(c["text"])

    try:
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer(
            "sentence-transformers/all-MiniLM-L6-v2", device="cpu")
        C = model.encode([c["text"] for c in records],
                         normalize_embeddings=True)
        Q = model.encode([query], normalize_embeddings=True)
        escores = (Q @ C.T)[0]
        print("Backend: embedding")
        for c, s in sorted(zip(records, escores), key=lambda x: -x[1]):
            print(c["id"], f"score={float(s):.3f}")
            print(c["text"])
    except Exception as error:
        print("Embedding backend unavailable:", error)

0 {'id': 'example:0', 'doc': 'example', 'start': 0, 'end': 49, 'text': 'Reset your password. Keep recovery codes private.'}
10 {'id': 'example:0', 'doc': 'example', 'start': 0, 'end': 49, 'text': 'Reset your password. Keep recovery codes private.'}
sentence {'id': 'example:0', 'doc': 'example', 'start': 0, 'end': 49, 'text': 'Reset your password. Keep recovery codes private.'}
Reference behaviour checks passed

Boundary tests:
empty text -> windows: [] | sentence: []
short document -> windows: [(0, 25)] | sentence: [(0, 25)]
invalid overlap -> ValueError: Require size > 0 and 0 <= overlap < size
95-word sentence -> sentence-aware fallback chunks: 3 sizes: [40, 40, 15]

Corpus word counts: {'handbook': 208, 'it_support': 192, 'library': 183}
strategy   chunks  min  max  emitted duplicated  rule+exception together
0              16    8   40      583          0  False (exception intact alone: False)
10             20   12   40      753        170  True (exception intact alone: True)
sent

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Backend: embedding
it:0 score=0.526
A forgotten password can be reset using the recovery link.
assessment:0 score=0.021
Assignments are uploaded through the learning portal.
